# Single-Band DRW Batch Fitting — All Galaxies
Fits a **Damped Random Walk (DRW)** Gaussian-Process model independently to every
available band for each galaxy in `GALAXY_LIST`.

**Bands fitted per galaxy (where data exists):**

| Band | Instrument | Pre-fit cuts |
|------|-----------|--------------|
| ZTF g | Zwicky Transient Facility | err < 0.5 mag, then 5σ MAD clip |
| ZTF r | Zwicky Transient Facility | err < 0.5 mag, then 5σ MAD clip |
| W1 | NEOWISE | Paper quality cuts, then 4σ MAD clip (Aravindan 2024 §2.1) |
| W2 | NEOWISE | Paper quality cuts, then 4σ MAD clip (Aravindan 2024 §2.1) |

> **Note:** `_bin_by_visit` is defined for visualization only — DRW fitting
> is always performed on the individual, quality-cut-and-sigma-clipped
> (unbinned) NEOWISE epochs.

**Key features:**
- `SKIP_COMPLETED=True` → resume a crashed run without re-fitting completed entries
- `SAVE_DIAGNOSTICS=True` → saves a DRW fit plot per (galaxy, band) as PNG
- `SAVE_NETCDF=True` → saves ArviZ `.nc` posterior per (galaxy, band) for offline replotting
- `SAVE_GALLERY=True` → saves per-galaxy 2×2 panel, traces, pair plots, and parameter chart
- CSV written after every individual fit — progress is never lost to a crash

**Reference:** Yu et al. 2025, *EzTaoX: Scalable & Robust AGN Light Curve Modeling*
([arXiv:2511.21479](https://arxiv.org/abs/2511.21479))


In [1]:
import warnings
warnings.filterwarnings("ignore")

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
import matplotlib.ticker as ticker
from matplotlib.lines import Line2D

import jax
import jax.numpy as jnp
jax.config.update("jax_enable_x64", True)

import tinygp
import numpyro
import numpyro.distributions as dist
from numpyro.infer import MCMC, NUTS, init_to_median
import arviz as az

from eztaox.kernels.quasisep import Exp
from eztaox.models import UniVarModel
from eztaox.fitter import random_search

# ── Matplotlib rcParams ──────────────────────────────────────────────────────
mpl.rcParams.update({
    'text.usetex': False,
    'axes.labelsize': 14,
    'xtick.labelsize': 11,
    'ytick.labelsize': 11,
})

# ── Wong (2011) colour-blind-safe palette ────────────────────────────────────
WONG_PALETTE = {
    'black':          '#000000',
    'orange':         '#E69F00',
    'sky_blue':       '#56B4E9',
    'bluish_green':   '#009E73',
    'yellow':         '#F0E442',
    'blue':           '#0072B2',
    'vermillion':     '#D55E00',
    'reddish_purple': '#CC79A7',
}

BAND_STYLE = {
    'ztf_g':      {'color': WONG_PALETTE['bluish_green'],   'marker': 'o',
                   'label': 'ZTF-g'},
    'ztf_r':      {'color': WONG_PALETTE['vermillion'],     'marker': 's',
                   'label': 'ZTF-r'},
    'neowise_w1': {'color': WONG_PALETTE['blue'],           'marker': 'o',
                   'label': r'NEOWISE W1 3.4 $\mu$m'},
    'neowise_w2': {'color': WONG_PALETTE['reddish_purple'], 'marker': 'o',
                   'label': r'NEOWISE W2 4.6 $\mu$m'},
}

DRW_LINE   = '#0B9E83'
DRW_FILL   = '#7FD8CB'
MEAN_COLOR = '#AAAAAA'

def _configure_font():
    import matplotlib.font_manager as fm
    available = {f.name for f in fm.fontManager.ttflist}
    if 'Avenir' in available:
        plt.rcParams['font.family'] = 'Avenir'
    else:
        fallbacks = ['Gill Sans', 'Helvetica Neue', 'Helvetica', 'Arial', 'DejaVu Sans']
        chosen = next((f for f in fallbacks if f in available), 'sans-serif')
        plt.rcParams['font.family'] = chosen

_configure_font()
print("JAX version :", jax.__version__)
print("Imports OK")


JAX version : 0.4.31
Imports OK


## 1. Utilities & Helper Functions

In [2]:
# ── Data-loading & sigma-clipping utilities ───────────────────────────────────

def load_ztf_band(filepath):
    """Load a ZTF single-band light curve from a whitespace/comma-delimited file.

    Expected columns: MJD, magnitude, magnitude_error.
    Returns arrays (t, mag, err) after removing bad epochs (err > 0.5 or NaN).
    """
    df = pd.read_csv(filepath, skipinitialspace=True)
    df.columns = df.columns.str.strip()
    cols = df.columns.tolist()
    t   = df[cols[0]].values.astype(float)
    mag = df[cols[1]].values.astype(float)
    err = df[cols[2]].values.astype(float)
    mask = np.isfinite(mag) & np.isfinite(err) & (err > 0) & (err < 0.5)
    return t[mask], mag[mask], err[mask]


def sigma_clip(t, mag, err, nsigma=4, n_iter=3):
    """Iterative sigma-clipping for ZTF data using median \u00b1 1.4826 \u00d7 MAD.

    Applied to ZTF bands (g and r).  NEOWISE data use paper quality
    cuts via load_neowise() and are sigma-clipped after loading.

    Parameters
    ----------
    nsigma : float  Clipping threshold (function default is 4; the batch
                    loop below calls this with nsigma=5 for ZTF and
                    nsigma=4 for NEOWISE).
    n_iter : int    Number of iterative passes (default 3).
    """
    mask = np.ones(len(mag), dtype=bool)
    for _ in range(n_iter):
        centre = np.median(mag[mask])
        scale  = 1.4826 * np.median(np.abs(mag[mask] - centre))
        mask   = np.abs(mag - centre) < nsigma * scale
    return t[mask], mag[mask], err[mask]


def load_neowise(neowise_path):
    """Load and quality-filter NEOWISE data for W1 and W2.

    Applies paper quality cuts (Aravindan 2024 \u00a72.1; Seacrest & Satyapal 2020).

    Cuts applied
    ------------
    qual_frame >= 10   : remove bad / fair-quality frames
    ph_qual[0] in {A,B}: W1 detection SNR > 3
    ph_qual[1] in {A,B}: W2 detection SNR > 3
    nb == 1            : single-bead fit; no source contamination
    na == 0            : no active deblending
    w2mpro < 14.5      : Eddington-bias cut (Seacrest & Satyapal 2020)

    Returns
    -------
    (t_w1, m_w1, e_w1), (t_w2, m_w2, e_w2)  -- each sorted by MJD
    """
    neo = pd.read_csv(neowise_path)
    neo.columns = neo.columns.str.strip()
    n_neo = len(neo)

    ph = neo['ph_qual'].astype(str)
    neo_ok = (
        (neo['qual_frame'] >= 10)   &
        ph.str[0].isin(['A', 'B'])  &
        ph.str[1].isin(['A', 'B'])  &
        (neo['nb']      == 1)       &
        (neo['na']      == 0)       &
        (neo['w2mpro']  < 14.5)
    )
    neo_f = neo[neo_ok]
    print(f'[NEOWISE]  {neo_ok.sum():3d}/{n_neo} rows passed quality cuts '
          f'(qual_frame\u226510, ph_qual A/B, nb==1, na==0, w2mpro<14.5).')

    def _sort(t, m, e):
        ok  = np.isfinite(m) & np.isfinite(e) & (e > 0)
        idx = np.argsort(t[ok])
        return t[ok][idx], m[ok][idx], e[ok][idx]

    t_w1, m_w1, e_w1 = _sort(
        neo_f['mjd'].values, neo_f['w1mpro'].values, neo_f['w1sigmpro'].values)
    t_w2, m_w2, e_w2 = _sort(
        neo_f['mjd'].values, neo_f['w2mpro'].values, neo_f['w2sigmpro'].values)

    print(f'           W1: {len(t_w1)} valid epochs  |  W2: {len(t_w2)} valid epochs')
    return (t_w1, m_w1, e_w1), (t_w2, m_w2, e_w2)


def _bin_by_visit(t, values, errors=None, gap_days=10):
    """Group single-epoch WISE data into ~10-day observing visits.

    FOR VISUALIZATION ONLY — never pass binned data to fit_drw_band.
    DRW fitting uses the individual quality-cut epochs from load_neowise().

    Returns (bin_mjd, bin_mean, bin_sem).  Single-epoch visits fall back
    to the reported photometric uncertainty.
    """
    t      = np.asarray(t,      dtype=float)
    values = np.asarray(values, dtype=float)
    errors = np.asarray(errors, dtype=float) if errors is not None else None

    order = np.argsort(t)
    ts, vs = t[order], values[order]
    es = errors[order] if errors is not None else None

    labels = np.zeros(len(ts), dtype=int)
    for i in range(1, len(ts)):
        labels[i] = labels[i-1] + int(ts[i] - ts[i-1] > gap_days)

    bt, bm, be = [], [], []
    for lbl in np.unique(labels):
        sel = labels == lbl
        n   = sel.sum()
        bt.append(ts[sel].mean())
        bm.append(vs[sel].mean())
        be.append(vs[sel].std(ddof=1) / np.sqrt(n) if n > 1
                  else (es[sel][0] if es is not None else 0.0))

    return np.array(bt), np.array(bm), np.array(be)


print("Utility functions loaded.")


Utility functions loaded.


In [3]:
# ── Variability statistics ────────────────────────────────────────────────────

def sigma_var_mmag(mag, err):
    """Excess-variance RMS in mmag (measurement noise subtracted)."""
    excess = mag.var(ddof=1) - (err**2).mean()
    return float(np.sqrt(max(0.0, excess))) * 1000.0


def mcmc_median_params(az_data):
    """Return (tau_days, sigma_mmag, mean_mag) posterior medians."""
    flat  = az_data.posterior.stack(sample=["chain", "draw"])
    tau   = float(np.median(np.exp(flat["log_drw_scale"].values)))
    sigma = float(np.median(np.exp(flat["log_drw_sigma"].values)))
    mean  = float(np.median(flat["mean"].values))
    return tau, sigma * 1000.0, mean


def mcmc_ci(az_data, param="log_drw_scale", ci=0.68):
    """68% credible interval on exp(param)."""
    flat  = az_data.posterior.stack(sample=["chain", "draw"])
    draws = np.exp(flat[param].values)
    lo, hi = np.percentile(draws, [(1 - ci) / 2 * 100, (1 + ci) / 2 * 100])
    return float(lo), float(hi)


# ── GP conditional prediction ─────────────────────────────────────────────────

def predict_drw(t_obs, m_obs, e_obs, tau, sigma_mag, mean_mag, t_pred):
    """GP conditional mean and +/-1-sigma standard deviation at t_pred."""
    idx    = np.argsort(t_obs)
    t_j    = jnp.array(t_obs[idx])
    m_j    = jnp.array(m_obs[idx]) - mean_mag
    d_j    = jnp.array(e_obs[idx])**2
    tp_j   = jnp.array(t_pred)
    k_fit  = Exp(scale=tau, sigma=sigma_mag)
    gp     = tinygp.GaussianProcess(k_fit, t_j, diag=d_j)
    _, cond = gp.condition(m_j, tp_j)
    return np.array(cond.loc) + mean_mag, np.sqrt(np.array(cond.variance))


# ── Main single-band DRW fitting routine ─────────────────────────────────────

def fit_drw_band(t, m, e, band_name, n_sample=10000, n_best=8,
                 n_warmup=1000, n_mcmc=5000, seed=0, progress_bar=True):
    """Fit a DRW model to a single-band light curve.

    Parameters
    ----------
    t, m, e      : 1-D ndarrays  MJD, magnitude, uncertainty (quality-cut).
    band_name    : str            Label shown in progress output.
    progress_bar : bool           Set False in batch mode to reduce output.

    Returns
    -------
    bestP   : dict                MLE parameter dict (from random_search).
    mcmc    : numpyro.infer.MCMC  Fitted MCMC object.
    az_data : arviz.InferenceData
    """
    idx = np.argsort(t)
    t_s = jnp.array(t[idx], dtype=jnp.float64)
    m_s = jnp.array(m[idx], dtype=jnp.float64)
    e_s = jnp.array(e[idx], dtype=jnp.float64)

    T      = float(t_s[-1] - t_s[0])
    dt_min = float(jnp.min(jnp.diff(t_s)))
    m_mean = float(jnp.mean(m_s))
    m_std  = float(jnp.std(m_s))

    log_scale_lo = np.log(max(dt_min * 0.5, 0.5))
    log_scale_hi = np.log(T * 10.0)

    k     = Exp(scale=100.0, sigma=m_std)
    model = UniVarModel(t_s, m_s, e_s, k, zero_mean=False)

    def init_sampler():
        log_s  = numpyro.sample("log_drw_scale",
                                dist.Uniform(log_scale_lo, log_scale_hi))
        log_a  = numpyro.sample("log_drw_sigma",
                                dist.Uniform(np.log(1e-3), np.log(2.0)))
        log_kp = jnp.stack([log_s, log_a])
        numpyro.deterministic("log_kernel_param", log_kp)
        mu = numpyro.sample("mean", dist.Normal(m_mean, m_std * 2.0))
        return {"log_kernel_param": log_kp, "mean": mu}

    print(f"\n{'='*55}")
    print(f"  {band_name}  -- MLE random search  (N={n_sample} samples)")
    print(f"{'='*55}")
    bestP, _ = random_search(
        model, init_sampler, jax.random.PRNGKey(seed), n_sample, n_best)
    tau_mle = np.exp(float(bestP["log_kernel_param"][0]))
    sig_mle = np.exp(float(bestP["log_kernel_param"][1])) * 1000
    print(f"  MLE -> tau_DRW = {tau_mle:.1f} d,  sigma_DRW = {sig_mle:.2f} mmag")

    def numpyro_model(m_obj):
        params = init_sampler()
        m_obj.sample(params)

    nuts     = NUTS(numpyro_model, dense_mass=True,
                    target_accept_prob=0.90, init_strategy=init_to_median)
    mcmc_obj = MCMC(nuts, num_warmup=n_warmup, num_samples=n_mcmc,
                    num_chains=5, progress_bar=progress_bar)

    print(f"  Running MCMC  (warmup={n_warmup}, samples={n_mcmc}) ...")
    mcmc_obj.run(jax.random.PRNGKey(seed + 1), model)

    az_data = az.from_numpyro(mcmc_obj)
    tau_med, sig_med, _ = mcmc_median_params(az_data)
    tau_lo, tau_hi = mcmc_ci(az_data, "log_drw_scale")
    print(f"  Posterior median -> tau_DRW = {tau_med:.1f} d  "
          f"[{tau_lo:.1f}, {tau_hi:.1f}]  "
          f"sigma_DRW = {sig_med:.2f} mmag")

    return bestP, mcmc_obj, az_data


print("DRW fitting helpers defined.")


DRW fitting helpers defined.


In [4]:
def plot_drw_fit(t_obs, m_obs, e_obs, az_data, band_label, ax=None, color=None, t0=None):
    """Plot data + DRW best-fit conditional mean / +/-1-sigma band.

    Parameters
    ----------
    t_obs, m_obs, e_obs : 1-D arrays  MJD, magnitude, uncertainty.
    az_data    : arviz.InferenceData  from fit_drw_band.
    band_label : str  shown in top-right corner.
    ax         : matplotlib Axes (created if None).
    color      : data-point colour (Wong vermillion if None).
    t0         : float or None  Common MJD origin for the time axis.  When
                 supplied (e.g. the global minimum across all bands in a
                 multi-panel gallery), the x-axis is shifted by this value
                 so that all panels share the same "Year 0".  Defaults to
                 t_obs.min() (i.e. the first observation of this band).
    """
    if ax is None:
        fig, ax = plt.subplots(figsize=(10, 5))
    if color is None:
        color = WONG_PALETTE['vermillion']

    tau, sig_mmag, mean_mag = mcmc_median_params(az_data)
    sig_mag = sig_mmag / 1000.0
    sv      = sigma_var_mmag(m_obs, e_obs)

    if t0 is None:
        t0 = t_obs.min()
    t_fine    = np.linspace(t_obs.min() - 30, t_obs.max() + 30, 800)
    t_fine_yr = (t_fine - t0) / 365.25
    t_obs_yr  = (t_obs  - t0) / 365.25

    pred_mean, pred_std = predict_drw(t_obs, m_obs, e_obs,
                                      tau, sig_mag, mean_mag, t_fine)

    ax.axhline(mean_mag, color=MEAN_COLOR, lw=0.9, ls="-", zorder=1)
    ax.fill_between(t_fine_yr,
                    pred_mean - pred_std, pred_mean + pred_std,
                    color=DRW_FILL, alpha=0.65, zorder=2)
    ax.plot(t_fine_yr, pred_mean, color=DRW_LINE, lw=1.8, zorder=3)
    ax.errorbar(t_obs_yr, m_obs, yerr=e_obs,
                fmt="o", ms=4.5, color=color,
                ecolor=color, elinewidth=0.75, capsize=0, zorder=4)

    if not ax.yaxis_inverted():
        ax.invert_yaxis()

    ax.tick_params(axis='both', direction='in', top=True, right=True, which='both')

    ann_txt = (
        f"$\\sigma_{{\\rm Var}}$ = {sv:.1f} mmag\n"
        f"$\\sigma_{{\\rm DRW}}$ = {sig_mmag:.1f} mmag"
    )
    ax.text(0.02, 0.96, ann_txt,
            transform=ax.transAxes, va="top", ha="left", fontsize=11,
            family="monospace",
            bbox=dict(boxstyle="round,pad=0.3", fc="white", alpha=0.7, lw=0))
    ax.text(0.98, 0.96, band_label,
            transform=ax.transAxes, va="top", ha="right",
            fontsize=15, fontweight="bold")

    ax.set_xlabel("Time (years)", fontsize=13)
    ax.set_ylabel("Magnitude",    fontsize=13)
    ax.set_xlim(left=-0.05)
    ax.grid(True, alpha=0.25, linestyle='--', linewidth=0.7)

    legend_handles = [
        Line2D([0], [0], color=DRW_LINE, lw=2.0, label="Best-fit DRW model"),
        Line2D([0], [0], marker="o", color="w", markerfacecolor=color,
               markersize=6, label="Data"),
    ]
    ax.legend(handles=legend_handles, loc="lower left", fontsize=10)
    return ax


print("plot_drw_fit() defined.")


plot_drw_fit() defined.


## 2. Batch Single-Band DRW Fitting

### Pre-fit data preparation
| Band | Method | Rationale |
|------|--------|-----------|
| ZTF g / r | 5$\sigma$ MAD sigma-clip (3 iterations) | Removes photometric artefacts from ZTF pipeline |
| NEOWISE W1 / W2 | Paper quality cuts and 4$\sigma$ MAD sigma-clip | qual_frame>=10, ph_qual A/B, nb==1, na==0, w2mpro<14.5 (Aravindan 2024 §2.1) |

> DRW fitting is performed on **individual epochs** after the above cuts.
> Visit-binning (`_bin_by_visit`) is available for visualization only.

### Output files per galaxy folder
| File | Contents |
|------|----------|
| `{galaxy}_{band}_drw.png` | DRW fit plot (if `SAVE_DIAGNOSTICS=True`) |
| `{galaxy}_{band}_posterior.nc` | ArviZ NetCDF posterior (if `SAVE_NETCDF=True`) |
| `{galaxy}_DRW_4panel.pdf` | 2×2 DRW-fit panel (if `SAVE_GALLERY=True`) |
| `{galaxy}_{band}_trace.pdf` | MCMC trace plots per band (if `SAVE_GALLERY=True`) |
| `{galaxy}_{band}_pair.pdf` | Posterior pair plots per band (if `SAVE_GALLERY=True`) |
| `{galaxy}_DRW_params_comparison.pdf` | τ / σ bar chart (if `SAVE_GALLERY=True`) |

### Running CSV
`drw_single_band_results.csv` is written after **every individual fit**.
Columns: `galaxy`, `band`, `band_label`, `n_obs`, `tau_drw_med`, `tau_drw_lo`,
`tau_drw_hi`, `sigma_drw_med` (mmag), `sigma_drw_lo`, `sigma_drw_hi`,
`sigma_var` (mmag), `n_eff_tau`, `status`.


In [5]:
# ── Paths ─────────────────────────────────────────────────────────────────────
BASE_DIR = "/path/to/light_curves"
SAVE_DIR = "/path/to/results/EzTaoX Plots/SingleBand"

# ── MCMC settings ─────────────────────────────────────────────────────────────
MCMC_WARMUP      = 1000    # warm-up steps per chain
MCMC_SAMPLES     = 5000    # posterior samples per chain

# ── Run options ───────────────────────────────────────────────────────────────
SAVE_DIAGNOSTICS = True    # save a DRW-fit PNG per (galaxy, band)
SAVE_NETCDF      = True    # save ArviZ .nc posterior per (galaxy, band)
SAVE_GALLERY     = True    # save 2x2 panel, trace, pair, param-comparison per galaxy
                           # (requires SAVE_NETCDF=True when using SKIP_COMPLETED,
                           #  so posteriors can be reloaded for already-completed bands)
SKIP_COMPLETED   = True# skip rows already marked 'ok' in the CSV (safe to rerun)
SHOW_PROGRESS    = False   # set True to see per-fit MCMC progress bars

# ── Single-band configs: (band_key, human_readable_label) ─────────────────────
_SINGLE_BAND_CONFIGS = [
    ('ztf_g',      'ZTF-g'),
    ('ztf_r',      'ZTF-r'),
    ('neowise_w1', 'NEOWISE W1'),
    ('neowise_w2', 'NEOWISE W2'),
]

# ── Galaxy list ───────────────────────────────────────────────────────────────
# Format: (name, ztf_g_file, ztf_r_file, neowise_file,
#           has_ztf_g, has_ztf_r, has_neowise)
# Set has_* = False for galaxies where that instrument file does not exist.
# Both NEOWISE W1 and W2 are extracted from the single neowise_file.
GALAXY_LIST = [
    ('J0847+0352',  'J0847+0352_ztf_g.txt',  'J0847+0352_ztf_r.txt',
     'J0847+0352_neowise.csv',   True,  True,  True),

    ('J0854+1737',  'J0854+1737_ztf_g.txt',  'J0854+1737_ztf_r.txt',
     'J0854+1737_neowise.csv',   True,  True,  True),

    ('J0928+1502',  'J0928+1502_ztf_g.txt',  'J0928+1502_ztf_r.txt',
     'J0928+1502_neowise.csv',   True,  True,  True),

    ('J0954+4032',  'J0954+4032_ztf_g.txt',  'J0954+4032_ztf_r.txt',
     'J0954+4032_neowise.csv',   True,  True,  True),

    ('J1032+2259',  'J1032+2259_ztf_g.txt',  'J1032+2259_ztf_r.txt',
     'J1032+2259_neowise.csv',   True,  True,  True),

    ('J1144+0726',  'J1144+0726_ztf_g.txt',  '',
     'J1144+0726_neowise.csv',   True,  False, True),

    ('J1205+4551',  'J1205+4551_ztf_g.txt',  '',
     'J1205+4551_neowise.csv',   True,  False, True),

    ('J1406-0019',  'J1406-0019_ztf_g.txt',  'J1406-0019_ztf_r.txt',
     'J1406-0019_neowise.csv',   True,  True,  True),

    ('J1428+5708',  'J1428+5708_ztf_g.txt',  'J1428+5708_ztf_r.txt',
     'J1428+5708_neowise.csv',   True,  True,  True),

    ('NSA10045',    'NSA10045_ztf_g.txt',    'NSA10045_ztf_r.txt',
     'NSA10045_neowise.csv',     True,  True,  True),

    ('NSA10779',    'NSA10779_ztf_g.txt',    'NSA10779_ztf_r.txt',
     'NSA10779_neowise.csv',     True,  True,  True),

    ('NSA115122',   'NSA115122_ztf_g.txt',   'NSA115122_ztf_r.txt',
     'NSA115122_neowise.csv',    True,  True,  True),

    ('NSA119259',   'NSA119259_ztf_g.txt',   'NSA119259_ztf_r.txt',
     'NSA119259_neowise.csv',    True,  True,  True),

    ('NSA125986',   'NSA125986_ztf_g.txt',   'NSA125986_ztf_r.txt',
     'NSA125986_neowise.csv',    True,  True,  True),

    ('NSA151888',   'NSA151888_ztf_g.txt',   'NSA151888_ztf_r.txt',
     'NSA151888_neowise.csv',    True,  True,  True),

    ('NSA165320',   'NSA165320_ztf_g.txt',   'NSA165320_ztf_r.txt',
     'NSA165320_neowise.csv',    True,  True,  True),

    ('NSA3462',     'NSA3462_ztf_g.txt',     'NSA3462_ztf_r.txt',
     'NSA3462_neowise.csv',      True,  True,  True),

    ('NSA3602',     'NSA3602_ztf_g.txt',     'NSA3602_ztf_r.txt',
     'NSA3602_neowise.csv',      True,  True,  True),

    ('NSA39952',    'NSA39952_ztf_g.txt',    'NSA39952_ztf_r.txt',
     'NSA39952_neowise.csv',     True,  True,  True),

    ('NSA56238',    'NSA56238_ztf_g.txt',    'NSA56238_ztf_r.txt',
     'NSA56238_neowise.csv',     True,  True,  True),

    ('NSA68133',    'NSA68133_ztf_g.txt',    'NSA68133_ztf_r.txt',
     'NSA68133_neowise.csv',     True,  True,  True),

    ('NSA74287',    'NSA74287_ztf_g.txt',    'NSA74287_ztf_r.txt',
     'NSA74287_neowise.csv',     True,  True,  True),

    ('NSA76203',    'NSA76203_ztf_g.txt',    'NSA76203_ztf_r.txt',
     'NSA76203_neowise.csv',     True,  True,  True),

    ('NSA79874',    'NSA79874_ztf_g.txt',    'NSA79874_ztf_r.txt',
     'NSA79874_neowise.csv',     True,  True,  True),

    ('NSA91579',    'NSA91579_ztf_g.txt',    'NSA91579_ztf_r.txt',
     'NSA91579_neowise.csv',     True,  True,  True),
]

print(f"Configuration loaded — {len(GALAXY_LIST)} galaxies, "
      f"{len(_SINGLE_BAND_CONFIGS)} bands each "
      f"({len(GALAXY_LIST) * len(_SINGLE_BAND_CONFIGS)} fits max).")


Configuration loaded — 25 galaxies, 4 bands each (100 fits max).


In [6]:
# ── Batch Single-Band DRW Fitting ─────────────────────────────────────────────
#
# For each galaxy in GALAXY_LIST x each available band:
#   1. Load + sigma-clip the light curve
#   2. Fit a single-band DRW (MLE random search -> MCMC)
#   3. Append tau_DRW, sigma_DRW, sigma_Var to the running CSV
#   4. Optionally save per-band PNG (SAVE_DIAGNOSTICS) and NetCDF (SAVE_NETCDF)
#   5. Optionally save per-galaxy gallery (SAVE_GALLERY):
#        • 2×2 DRW-fit panel PDF
#        • MCMC summary table (printed)
#        • Trace plots PDF per band
#        • Posterior pair plots PDF per band
#        • DRW parameter summary table (printed)
#        • τ_DRW / σ_DRW comparison bar chart PDF

# ── Resume: load any existing results ─────────────────────────────────────────
results_path = os.path.join(SAVE_DIR, 'drw_single_band_results.csv')
if SKIP_COMPLETED and os.path.exists(results_path):
    _existing   = pd.read_csv(results_path)
    all_results = _existing.to_dict('records')
    _done       = set(zip(_existing['galaxy'], _existing['band']))
    print(f"Resuming -- {len(_done)} (galaxy, band) fits already completed.")
else:
    all_results = []
    _done       = set()

os.makedirs(SAVE_DIR, exist_ok=True)
n_ok_total  = 0
n_err_total = 0

# ── Main loop ─────────────────────────────────────────────────────────────────
for (gname, g_file, r_file, neo_file,
     has_g, has_r, has_neowise) in GALAXY_LIST:

    print(f'\n{"="*60}')
    print(f'Galaxy: {gname}')

    galaxy_dir = os.path.join(SAVE_DIR, gname)
    os.makedirs(galaxy_dir, exist_ok=True)
    n_ok_gal = 0

    # ── Load all available bands ───────────────────────────────────────────────
    _gal_bands = {}
    try:
        if has_g and g_file:
            tg, mg, eg = load_ztf_band(os.path.join(BASE_DIR, g_file))
            tg, mg, eg = sigma_clip(tg, mg, eg, nsigma=5)
            if len(tg) >= 5:
                _gal_bands['ztf_g'] = (tg, mg, eg)
            else:
                print(f'  [ZTF-g]  Only {len(tg)} epochs after cuts -- skipping.')

        if has_r and r_file:
            tr, mr, er = load_ztf_band(os.path.join(BASE_DIR, r_file))
            tr, mr, er = sigma_clip(tr, mr, er, nsigma=5)
            if len(tr) >= 5:
                _gal_bands['ztf_r'] = (tr, mr, er)
            else:
                print(f'  [ZTF-r]  Only {len(tr)} epochs after cuts -- skipping.')

        if has_neowise and neo_file:
            # Paper quality cuts (load_neowise), then optional second-pass sigma-clip
            (tw1, mw1, ew1), (tw2, mw2, ew2) = load_neowise(
                os.path.join(BASE_DIR, neo_file))
            tw1, mw1, ew1 = sigma_clip(tw1, mw1, ew1, nsigma=4)
            tw2, mw2, ew2 = sigma_clip(tw2, mw2, ew2, nsigma=4)
            if len(tw1) >= 5:
                _gal_bands['neowise_w1'] = (tw1, mw1, ew1)
            else:
                print(f'  [NEOWISE W1]  Only {len(tw1)} epochs after quality cuts -- skipping.')
            if len(tw2) >= 5:
                _gal_bands['neowise_w2'] = (tw2, mw2, ew2)
            else:
                print(f'  [NEOWISE W2]  Only {len(tw2)} epochs after quality cuts -- skipping.')

    except Exception as exc:
        print(f'  LOAD ERROR: {exc}')
        all_results.append({'galaxy': gname, 'band': 'all', 'band_label': 'all',
                            'n_obs': 0, 'status': f'load error: {exc}'})
        pd.DataFrame(all_results).to_csv(results_path, index=False)
        n_err_total += 1
        continue

    print(f'  Available bands: {list(_gal_bands.keys())}')

    # ── Fit each band ──────────────────────────────────────────────────────────
    # band_results collects successful fits for the per-galaxy gallery.
    # For SKIP_COMPLETED runs the previously-fitted bands are reloaded from their
    # NetCDF posteriors so the gallery is always complete.
    band_results = []   # (t, m, e, az_data, band_key, band_label)

    for band_key, band_label in _SINGLE_BAND_CONFIGS:
        if band_key not in _gal_bands:
            continue

        t, m, e = _gal_bands[band_key]

        # ── Already-completed band: reload posterior for gallery ─────────────
        if SKIP_COMPLETED and (gname, band_key) in _done:
            print(f'  [{band_label}] already done -- skipping fit')
            if SAVE_GALLERY:
                nc_path = os.path.join(galaxy_dir,
                                       f'{gname}_{band_key}_posterior.nc')
                if os.path.exists(nc_path):
                    try:
                        az_reload = az.from_netcdf(nc_path)
                        band_results.append((t, m, e, az_reload,
                                             band_key, band_label))
                        print(f'    Posterior reloaded from NetCDF for gallery.')
                    except Exception as rel_exc:
                        print(f'    Could not reload NetCDF: {rel_exc}')
                else:
                    print(f'    No NetCDF found at {nc_path} -- '
                          f'band omitted from gallery.')
            continue

        # ── Fit this band ────────────────────────────────────────────────────
        row = {'galaxy': gname, 'band': band_key, 'band_label': band_label,
               'n_obs': len(t), 'status': 'error'}

        try:
            print(f'\n  [{band_label}]  N = {len(t)} epochs')

            bestP, mcmc_obj, az_data = fit_drw_band(
                t, m, e,
                band_name    = f'{gname} {band_label}',
                n_warmup     = MCMC_WARMUP,
                n_mcmc       = MCMC_SAMPLES,
                seed         = 42,
                progress_bar = SHOW_PROGRESS,
            )

            # ── Extract posterior summaries ──────────────────────────────────
            tau_med,  sig_med,  _ = mcmc_median_params(az_data)
            tau_lo,   tau_hi      = mcmc_ci(az_data, 'log_drw_scale')
            sig_lo = mcmc_ci(az_data, 'log_drw_sigma')[0] * 1000   # mmag
            sig_hi = mcmc_ci(az_data, 'log_drw_sigma')[1] * 1000   # mmag
            sv     = sigma_var_mmag(m, e)
            n_eff  = int(
                az.summary(az_data, var_names=['log_drw_scale'])['ess_bulk'].values[0]
            )

            row.update({
                'tau_drw_med':   round(tau_med, 2),
                'tau_drw_lo':    round(tau_lo,  2),
                'tau_drw_hi':    round(tau_hi,  2),
                'sigma_drw_med': round(sig_med, 3),
                'sigma_drw_lo':  round(sig_lo,  3),
                'sigma_drw_hi':  round(sig_hi,  3),
                'sigma_var':     round(sv,       3),
                'n_eff_tau':     n_eff,
                'status':        'ok',
            })

            print(f'  -> tau = {tau_med:.1f} [{tau_lo:.1f}, {tau_hi:.1f}] d  '
                  f'sigma_DRW = {sig_med:.2f} mmag  '
                  f'sigma_Var = {sv:.2f} mmag  '
                  f'n_eff = {n_eff}')
            n_ok_gal += 1

            # ── Save NetCDF posterior ────────────────────────────────────────
            if SAVE_NETCDF:
                nc_path = os.path.join(
                    galaxy_dir, f'{gname}_{band_key}_posterior.nc')
                az.to_netcdf(az_data, nc_path)

            # ── Save per-band diagnostic DRW-fit plot ────────────────────────
            if SAVE_DIAGNOSTICS:
                style   = BAND_STYLE[band_key]
                fig, ax = plt.subplots(figsize=(11, 5))
                plot_drw_fit(t, m, e, az_data,
                             band_label = style['label'],
                             ax=ax, color=style['color'])
                ax.set_title(f'{gname}  --  {style["label"]}',
                             fontsize=14, pad=8)
                fig.tight_layout()
                plot_path = os.path.join(
                    galaxy_dir, f'{gname}_{band_key}_drw.png')
                plt.savefig(plot_path, dpi=150, bbox_inches='tight')
                plt.close(fig)

            # ── Collect for per-galaxy gallery ───────────────────────────────
            band_results.append((t, m, e, az_data, band_key, band_label))

        except Exception as exc:
            row['status'] = f'error: {exc}'
            print(f'  [{band_label}] ERROR: {exc}')
            n_err_total += 1

        all_results.append(row)
        pd.DataFrame(all_results).to_csv(results_path, index=False)

    # ── Per-galaxy gallery (runs after all bands are fitted) ───────────────────
    if SAVE_GALLERY and len(band_results) >= 1:
        n_fitted = len(band_results)
        print(f'\n  Generating gallery for {gname}  ({n_fitted} band(s)) ...')

        # 1. 2×2 DRW-fit panel PDF
        fig, axes = plt.subplots(2, 2, figsize=(18, 10), constrained_layout=True)
        fig.suptitle(f"{gname} \u2014 Single-Band DRW Fits",
                     fontsize=17, fontweight="bold")
        flat_axes = axes.flatten()
        # Use a single time origin across all bands so the x-axes are
        # directly comparable (e.g. ZTF and NEOWISE both start at Year 0).
        t0_global = min(t.min() for t, m, e, az_d, bkey, _ in band_results)

        for ax, (t, m, e, az_d, bkey, _) in zip(flat_axes, band_results):
            style = BAND_STYLE[bkey]
            plot_drw_fit(t, m, e, az_d, style['label'], ax=ax, color=style['color'], t0=t0_global)
        for ax in flat_axes[n_fitted:]:    # hide unused panels
            ax.set_visible(False)
        # Synchronise x-axis limits across all fitted panels so that
        # bands are time-aligned. Different sources start at different
        # MJDs, but t0_global already anchors all panels to the same
        # origin; this step also forces the right edge to be identical
        # so no panel is cut short or padded differently.
        _xlims = [ax.get_xlim() for ax in flat_axes[:n_fitted]]
        _x_lo  = min(xl[0] for xl in _xlims)
        _x_hi  = max(xl[1] for xl in _xlims)
        for ax in flat_axes[:n_fitted]:
            ax.set_xlim(_x_lo, _x_hi)
        panel_path = os.path.join(galaxy_dir, f'{gname}_DRW_4panel.pdf')
        plt.savefig(panel_path, bbox_inches='tight', dpi=150)
        plt.close(fig)
        print(f'  Saved {panel_path}')

        # 2. MCMC summary tables (printed to notebook output)
        print(f"\n  {'-'*50}")
        print(f"  {gname} \u2014 MCMC Summaries")
        print(f"  {'-'*50}")
        for _, _, _, az_d, bkey, _ in band_results:
            band_lbl = BAND_STYLE[bkey]['label']
            print(f"\n  {band_lbl} \u2014 MCMC summary")
            sum_df = az.summary(
                az_d, var_names=["log_drw_scale", "log_drw_sigma", "mean"])
            print(sum_df[["mean", "sd", "hdi_3%", "hdi_97%",
                          "r_hat", "ess_bulk"]].to_string())

        # 3. Trace plots (one PDF per band)
        for _, _, _, az_d, bkey, _ in band_results:
            band_lbl = BAND_STYLE[bkey]['label']
            az.plot_trace(
                az_d,
                var_names=["log_drw_scale", "log_drw_sigma", "mean"],
                figsize=(12, 6),
            )
            plt.suptitle(f"{gname} {band_lbl} \u2014 Trace Plots",
                         fontsize=13, y=1.01)
            plt.tight_layout()
            trace_path = os.path.join(galaxy_dir, f'{gname}_{bkey}_trace.pdf')
            plt.savefig(trace_path, bbox_inches='tight', dpi=150)
            plt.close()

        # 4. Posterior pair plots (one PDF per band)
        for _, _, _, az_d, bkey, _ in band_results:
            band_lbl = BAND_STYLE[bkey]['label']
            az.plot_pair(
                az_d,
                var_names=["log_drw_scale", "log_drw_sigma"],
                kind="scatter",
                marginals=True,
                figsize=(6, 6),
            )
            plt.suptitle(f"{gname} {band_lbl} \u2014 Posterior Pair Plot",
                         fontsize=12, y=0.94)
            plt.tight_layout()
            pair_path = os.path.join(galaxy_dir, f'{gname}_{bkey}_pair.pdf')
            plt.savefig(pair_path, bbox_inches='tight', dpi=150)
            plt.close()

        # 5. DRW parameter summary table (printed to notebook output)
        rows_gal = []
        for t, m, e, az_d, bkey, _ in band_results:
            band_lbl            = BAND_STYLE[bkey]['label']
            tau_med, sig_med, _ = mcmc_median_params(az_d)
            tau_lo, tau_hi      = mcmc_ci(az_d, "log_drw_scale")
            sig_lo              = mcmc_ci(az_d, "log_drw_sigma")[0] * 1000
            sig_hi              = mcmc_ci(az_d, "log_drw_sigma")[1] * 1000
            sv                  = sigma_var_mmag(m, e)
            n_eff               = int(
                az.summary(az_d, var_names=["log_drw_scale"])["ess_bulk"].values[0]
            )
            rows_gal.append({
                "Band"               : band_lbl,
                "N_obs"              : len(t),
                "\u03c4_DRW [d]"     : f"{tau_med:.0f}",
                "\u03c4 CI [d]"      : f"[{tau_lo:.0f}, {tau_hi:.0f}]",
                "\u03c3_DRW [mmag]"  : f"{sig_med:.1f}",
                "\u03c3 CI [mmag]"   : f"[{sig_lo:.1f}, {sig_hi:.1f}]",
                "\u03c3_Var [mmag]"  : f"{sv:.1f}",
                "n_eff (\u03c4)"     : f"{n_eff}",
            })
        gal_summary = pd.DataFrame(rows_gal).set_index("Band")
        print(f"\n  {gname} \u2014 DRW Parameters Summary")
        print(gal_summary.to_string())

        # 6. τ_DRW / σ_DRW comparison bar chart PDF
        labels_bar, taus, tau_los_bar, tau_his_bar, sigs_bar, colors_bar = \
            [], [], [], [], [], []
        for t, m, e, az_d, bkey, _ in band_results:
            tau_med, sig_med, _ = mcmc_median_params(az_d)
            lo, hi              = mcmc_ci(az_d, "log_drw_scale")
            labels_bar.append(BAND_STYLE[bkey]['label'])
            taus.append(tau_med)
            tau_los_bar.append(tau_med - lo)
            tau_his_bar.append(hi - tau_med)
            sigs_bar.append(sig_med)
            colors_bar.append(BAND_STYLE[bkey]['color'])

        x = np.arange(len(labels_bar))
        fig, ax_bar = plt.subplots(1, 2, figsize=(13, 5))

        ax_bar[0].bar(x, taus, yerr=[tau_los_bar, tau_his_bar], color=colors_bar,
                      capsize=6, edgecolor='k', linewidth=0.8,
                      error_kw={"elinewidth": 1.5})
        ax_bar[0].set_xticks(x)
        ax_bar[0].set_xticklabels(labels_bar, fontsize=9)
        ax_bar[0].set_ylabel(r"$\tau$_DRW (days)", fontsize=13)
        ax_bar[0].set_title("DRW Timescale", fontsize=13)
        ax_bar[0].set_yscale("log")
        ax_bar[0].yaxis.set_major_formatter(ticker.ScalarFormatter())
        ax_bar[0].tick_params(axis='both', direction='in', top=True, right=True)

        ax_bar[1].bar(x, sigs_bar, color=colors_bar, edgecolor='k', linewidth=0.8)
        ax_bar[1].set_xticks(x)
        ax_bar[1].set_xticklabels(labels_bar, fontsize=9)
        ax_bar[1].set_ylabel(r"$\sigma$_DRW (mmag)", fontsize=13)
        ax_bar[1].set_title("DRW Variability Amplitude", fontsize=13)
        ax_bar[1].tick_params(axis='both', direction='in', top=True, right=True)

        fig.suptitle(f"{gname} \u2014 DRW Parameters by Band",
                     fontsize=14, fontweight="bold")
        fig.tight_layout()
        comp_path = os.path.join(galaxy_dir, f'{gname}_DRW_params_comparison.pdf')
        plt.savefig(comp_path, bbox_inches='tight', dpi=150)
        plt.close(fig)
        print(f'  Saved {comp_path}')

    n_ok_total += n_ok_gal
    print(f'\n  {gname}: {n_ok_gal} / {len(_gal_bands)} bands fitted OK.')

# ── Final summary ──────────────────────────────────────────────────────────────
results_df = pd.DataFrame(all_results)
results_df.to_csv(results_path, index=False)
print(f'\n{"="*60}')
print(f'Batch complete.  {n_ok_total} fits OK,  {n_err_total} errors.')
print(f'Results saved -> {results_path}')
print(f'{"="*60}')

print("DONE!")



Galaxy: J0847+0352
[NEOWISE]  260/281 rows passed quality cuts (qual_frame≥10, ph_qual A/B, nb==1, na==0, w2mpro<14.5).
           W1: 260 valid epochs  |  W2: 260 valid epochs
  Available bands: ['ztf_g', 'ztf_r', 'neowise_w1', 'neowise_w2']

  [ZTF-g]  N = 419 epochs

  J0847+0352 ZTF-g  -- MLE random search  (N=10000 samples)
  MLE -> tau_DRW = 1227.2 d,  sigma_DRW = 6.52 mmag
  Running MCMC  (warmup=1000, samples=5000) ...
  Posterior median -> tau_DRW = 4351.6 d  [918.1, 15683.7]  sigma_QSO = 10.98 mmag
  -> tau = 4351.6 [918.1, 15683.7] d  sigma_QSO = 10.98 mmag  sigma_Var = 0.00 mmag  n_eff = 2520

  [ZTF-r]  N = 151 epochs

  J0847+0352 ZTF-r  -- MLE random search  (N=10000 samples)
  MLE -> tau_DRW = 289.0 d,  sigma_DRW = 9.30 mmag
  Running MCMC  (warmup=1000, samples=5000) ...
  Posterior median -> tau_DRW = 809.7 d  [86.4, 7324.9]  sigma_QSO = 10.71 mmag
  -> tau = 809.7 [86.4, 7324.9] d  sigma_QSO = 10.71 mmag  sigma_Var = 0.00 mmag  n_eff = 4845

  [NEOWISE W1]  N = 260 

## 3. Results Summary

In [7]:
# ── Load results CSV and display ─────────────────────────────────────────────
results_df = pd.read_csv(os.path.join(SAVE_DIR, 'drw_single_band_results.csv'))

ok   = results_df[results_df['status'] == 'ok'].copy()
fail = results_df[results_df['status'] != 'ok']

print(f"Total fits:  {len(results_df)}")
print(f"  OK:        {len(ok)}")
print(f"  Failed:    {len(fail)}")
if len(fail):
    print("\nFailed entries:")
    print(fail[['galaxy', 'band', 'status']].to_string(index=False))

# ── Pivot table: tau_DRW per galaxy x band ────────────────────────────────────
if len(ok):
    pivot = ok.pivot_table(
        index='galaxy',
        columns='band',
        values='tau_drw_med',
        aggfunc='first',
    )
    print("\ntau_DRW (days) by galaxy and band:")
    print(pivot.to_string())
    print()

ok


Total fits:  96
  OK:        96
  Failed:    0

tau_DRW (days) by galaxy and band:
band        neowise_w1  neowise_w2    ztf_g    ztf_r
galaxy                                              
J0847+0352     9847.45    12616.92  4351.61   809.72
J0854+1737      100.77      619.38    37.34    66.25
J0928+1502     8702.16     9550.98   616.49    82.31
J0954+4032     2457.40     3687.85   106.23    44.87
J1032+2259     1020.33      371.52   428.45    52.98
J1144+0726     8352.93     5876.82   572.05      NaN
J1205+4551     7869.15     6714.69   408.17      NaN
J1406-0019      460.25      300.95   241.95  2259.73
J1428+5708     3895.91     9225.35  2710.27   394.24
NSA10045          6.50      787.76    52.58    58.63
NSA10779      12412.88    12422.74   224.27    80.85
NSA115122      8121.78     5059.49  4909.90     2.89
NSA119259          NaN         NaN   351.24   180.41
NSA125986       883.66     2876.20    30.88    10.95
NSA151888     10284.81    13690.40  3474.71   294.28
NSA165320       

,galaxy,band,band_label,n_obs,status,tau_drw_med,tau_drw_lo,tau_drw_hi,sigma_drw_med,sigma_drw_lo,sigma_drw_hi,sigma_var,n_eff_tau
0,J0847+0352,ztf_g,ZTF-g,419,ok,4351.61,918.11,15683.75,10.975,5.440,22.090,0.000,2520
1,J0847+0352,ztf_r,ZTF-r,151,ok,809.72,86.45,7324.90,10.714,3.770,26.732,0.000,4845
2,J0847+0352,neowise_w1,NEOWISE W1,260,ok,9847.45,3554.55,23207.18,132.938,82.563,205.098,90.574,4233
3,J0847+0352,neowise_w2,NEOWISE W2,260,ok,12616.92,5158.45,25726.58,150.286,98.051,217.535,102.526,4892
4,J0854+1737,ztf_g,ZTF-g,456,ok,37.34,28.83,50.64,65.780,59.235,74.871,60.651,15288
...,...,...,...,...,...,...,...,...,...,...,...,...,...
91,NSA79874,neowise_w2,NEOWISE W2,252,ok,5.66,0.68,684.07,84.807,66.903,113.047,91.850,1023
92,NSA91579,ztf_g,ZTF-g,850,ok,9.03,5.89,16.68,28.388,24.236,32.618,0.000,6085
93,NSA91579,ztf_r,ZTF-r,171,ok,638.49,14.19,9556.20,6.652,2.025,16.369,0.000,4988
94,NSA91579,neowise_w1,NEOWISE W1,147,ok,3441.95,1089.02,13518.16,287.002,168.985,553.558,151.079,3754
